## Imports

Pulls in `tensor` (used to convert Python lists into PyTorch tensors) and PyTorch's `Dataset`, `DataLoader`, and `random_split` utilities, which together handle storing, splitting, and batching the modular-arithmetic data.

In [ ]:
from torch import long, tensor
from torch.utils.data import Dataset, DataLoader, random_split

## Generating all `(a, b, target)` pairs

`generate_pairs(number)` builds every possible pair `(a, b)` with `a, b` in `range(number)`, together with the correct answer `(a + b) % number`. For `number=97` this produces all 9409 addition facts of the `(a+b) mod 97` task as a plain list.

In [ ]:
def generate_pairs(number):
    pairs = []
    for i in range(number):
        for j in range(number):
            pairs.append((i, j, (i + j) % number))
    return pairs

## Building train/test DataLoaders

`get_dataloaders` wraps the full `ModularArithmeticDataset` in PyTorch's `random_split` to carve off a **30% train / 70% test** split (a small train fraction is what causes the grokking phenomenon), then wraps each half in a `DataLoader` — the train loader shuffles each epoch, the test loader does not.

In [ ]:
def get_dataloaders(number, batch_size):
    modular_arithmetic_dataset = ModularArithmeticDataset(number)
    train_size = int(0.3 * len(modular_arithmetic_dataset))
    test_size = len(modular_arithmetic_dataset) - train_size

    train_dataset, test_dataset = random_split(modular_arithmetic_dataset, [train_size, test_size])
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

    return train_dataloader, test_dataloader

## The `ModularArithmeticDataset` class

A `torch.utils.data.Dataset` subclass that stores all generated pairs once at construction. `__getitem__` returns `(input_tensor, target)`, where `get_tensor` builds the input as the token sequence `[a, b, "="]` (with `97` used as the special `"="` token id) — the target answer is kept separate so it never leaks into the model's input.

In [ ]:
class ModularArithmeticDataset(Dataset):
    def __init__(self, number):
        self.pairs = generate_pairs(number)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return (self.get_tensor(self.pairs[idx]), self.pairs[idx][2])
    
    def get_tensor(self, item):
        sequence = [item[0], item[1], 97]
        return tensor(sequence)

## Manual sanity check

A leftover `if __name__ == "__main__":` block that just prints a hardcoded example tensor `[5, 3, 8, 97]` — a quick manual check of what a token sequence looks like, not tied to the actual dataset class.

In [ ]:
if __name__ == "__main__":
    print(tensor([5, 3, 8, 97]))